# Huấn luyện Mô hình Nhận diện Cử chỉ Tay riêng cho Phòng Thí Nghiệm Ảo
Sổ tay Google Colab huấn luyện mô hình MediaPipe Gesture Recognizer bằng `mediapipe-model-maker`.

### Các cử chỉ riêng trong phòng lab:
1. `cam_ong_nghiem`: Cầm ống nghiệm thẳng đứng
2. `rot_chat_long`: Nghiêng bàn tay rót hóa chất
3. `van_khoa_buret`: Chụm 3 ngón tay xoay núm vặn
4. `gio_hai_ngon`: Giơ ngón trỏ + ngón giữa (chữ V) để chọn menu
5. `none`: Nền / tư thế tự nhiên không thao tác

Đầu ra sẽ xuất tệp: `gesture_recognizer_lab.task`.

## 1. Cài đặt thư viện `mediapipe-model-maker`

In [ ]:
!pip install -q mediapipe-model-maker tensorflow

## 2. Nén và tải thư mục dữ liệu đã thu thập từ máy tính
Trên máy tính sau khi chạy `.venv/bin/python tools/thu_cu_chi.py`, nén thư mục `data/du_lieu_cu_chi` thành `du_lieu_cu_chi.zip` và upload lên đây.

In [ ]:
from google.colab import files
import zipfile
import os

print("Tải lên tệp du_lieu_cu_chi.zip:")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn, 'r') as zip_ref:
            zip_ref.extractall('dataset_cu_chi')
        print(f"Đã giải nén vào dataset_cu_chi/")

## 3. Nạp dữ liệu và phân chia tập Train / Val / Test

In [ ]:
from mediapipe_model_maker import gesture_recognizer

dataset_path = 'dataset_cu_chi'
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)

train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)
print(f"Dữ liệu: {len(train_data)} train, {len(validation_data)} val, {len(test_data)} test")

## 4. Huấn luyện mô hình Gesture Recognizer

In [ ]:
hparams = gesture_recognizer.HParams(export_dir="xuat_mo_hinh", epochs=20, batch_size=16)
model_options = gesture_recognizer.ModelOptions(dropout_rate=0.2)
options = gesture_recognizer.GestureRecognizerOptions(model_options=model_options, hparams=hparams)

model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

loss, acc = model.evaluate(test_data, batch_size=16)
print(f"Độ chính xác kiểm thử (Test accuracy): {acc * 100:.2f}%")

## 5. Xuất tệp `gesture_recognizer_lab.task` và tải về máy tính

In [ ]:
model.export_model()
output_file = "xuat_mo_hinh/gesture_recognizer.task"
target_file = "gesture_recognizer_lab.task"

if os.path.exists(output_file):
    os.rename(output_file, target_file)
    print(f"Đã xuất {target_file} thành công!")
    files.download(target_file)
    print("Sau khi tải về, hãy chép tệp vào thư mục 'tools/mo_hinh/' trong dự án.")
else:
    print("Lỗi: Không tìm thấy tệp mô hình đã xuất.")